# Neural Network Heuristic for Tic-Tac-Toe
## Learning Board Evaluation with PyTorch

This notebook trains a neural network to evaluate Tic-Tac-Toe board positions,
replacing the handcrafted heuristic used in depth-limited Alpha-Beta search.

**Key idea:** Instead of manually coding evaluation rules, we train a small
feedforward network on ground-truth minimax scores so the network *learns*
what makes a position good or bad.

---

## 1. Setup
Clone the repository and install dependencies.

In [ ]:
# Uncomment these lines if running on Google Colab
# !git clone -b claude/tic-tac-toe-ai-game-8TPHb https://github.com/amanverma-wsu/440-Project.git
# %cd 440-Project
# !pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import random
import time
import os

from board import Board
from ai import MinimaxAgent, AlphaBetaAgent, RandomAgent
from heuristic import evaluate_board

print(f"PyTorch {torch.__version__}")
print(f"NumPy {np.__version__}")
print("All modules loaded successfully!")

---
## 2. Board Representation Review
Quick refresher on the Board class before we encode states for the neural network.

In [ ]:
# Board basics
demo = Board(3)
demo.make_move(0, 0, Board.X)
demo.make_move(1, 1, Board.O)
demo.make_move(0, 1, Board.X)
print("Sample board state:")
print(demo)
print(f"\nCurrent player: {demo.current_player()}")
print(f"Empty cells: {demo.get_empty_cells()}")
print(f"Winner: {demo.check_winner()}")

---
## 3. Training Data Generation

We enumerate **all reachable 3×3 board states** and label each with its true
minimax score. This gives us ground-truth labels — the neural network will
learn to predict these scores.

**Board encoding:** Each cell maps to `X=+1, O=-1, empty=0`, flattened to a
9-element vector plus a turn indicator (`+1` if it's AI's turn, `-1` otherwise)
= **10 input features**.

In [ ]:
from nn_heuristic import encode_board, generate_training_data

# Show encoding for the demo board
encoded = encode_board(demo, Board.X, Board.O)
print("Board:")
print(demo)
print(f"\nEncoded: {encoded}")
print(f"Shape: {encoded.shape}  (9 cells + 1 turn indicator)")

In [ ]:
# Generate all training data (takes ~10-30 seconds)
print("Generating training data from all reachable 3x3 states...")
start = time.time()
X_data, y_data = generate_training_data(board_size=3)
elapsed = time.time() - start

print(f"\nGenerated {len(X_data)} labeled board states in {elapsed:.1f}s")
print(f"Input shape: {X_data.shape}")
print(f"Label range: [{y_data.min():.2f}, {y_data.max():.2f}] (normalized minimax scores)")

In [ ]:
# Visualize the distribution of minimax scores
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_data, bins=30, color='#3498db', edgecolor='black', alpha=0.8)
ax.set_xlabel('Normalized Minimax Score')
ax.set_ylabel('Number of Board States')
ax.set_title('Distribution of Ground-Truth Minimax Scores (3×3)')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='Draw (score=0)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nPositive scores (X winning): {(y_data > 0).sum()}")
print(f"Zero scores (draws):         {(y_data == 0).sum()}")
print(f"Negative scores (O winning): {(y_data < 0).sum()}")

---
## 4. Neural Network Architecture

We use a simple feedforward network:

```
Input (10) → Linear(64) → ReLU → Linear(32) → ReLU → Linear(1) → Tanh
```

- **Input**: 10 features (9 board cells + 1 turn indicator)
- **Hidden layers**: 64 and 32 neurons with ReLU activation
- **Output**: Single scalar in [-1, 1] via tanh (scaled to [-10, 10] at inference)
- **Loss**: Mean Squared Error vs true minimax scores

In [ ]:
from nn_heuristic import BoardNet

model = BoardNet(input_size=10)
print("BoardNet Architecture:")
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## 5. Training

Train the network with **Adam optimizer** and **MSE loss**.
- 80/20 train/validation split
- Batch size: 64
- 500 epochs (converges quickly on this small dataset)

In [ ]:
# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Train/validation split
n = len(X_data)
indices = np.random.permutation(n)
split = int(0.8 * n)
train_idx, val_idx = indices[:split], indices[split:]

X_train = torch.tensor(X_data[train_idx])
y_train = torch.tensor(y_data[train_idx]).unsqueeze(1)
X_val = torch.tensor(X_data[val_idx])
y_val = torch.tensor(y_data[val_idx]).unsqueeze(1)

print(f"Training set:   {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")

In [ ]:
# Training loop
model = BoardNet(input_size=10)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

epochs = 500
train_losses = []
val_losses = []

print(f"Training for {epochs} epochs...\n")
start_time = time.time()

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    epoch_loss /= len(X_train)
    train_losses.append(epoch_loss)

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = loss_fn(val_pred, y_val).item()
    val_losses.append(val_loss)

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch + 1:>3}/{epochs} — train: {epoch_loss:.6f}, val: {val_loss:.6f}")

training_time = time.time() - start_time
print(f"\nTraining complete in {training_time:.1f}s")
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss:   {val_losses[-1]:.6f}")

# Save weights
os.makedirs('results', exist_ok=True)
torch.save(model.state_dict(), 'results/nn_weights_3x3.pt')
print("\nModel saved to results/nn_weights_3x3.pt")

In [ ]:
# Plot training loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(train_losses, label='Train', color='#3498db', linewidth=1)
ax1.plot(val_losses, label='Validation', color='#e74c3c', linewidth=1)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.set_title('Training Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Zoomed view (skip first 10 epochs)
ax2.plot(range(10, epochs), train_losses[10:], label='Train', color='#3498db', linewidth=1)
ax2.plot(range(10, epochs), val_losses[10:], label='Validation', color='#e74c3c', linewidth=1)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MSE Loss')
ax2.set_title('Training Loss (after epoch 10)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Accuracy Evaluation

Compare the neural network's predictions against:
1. **Ground truth** minimax scores
2. **Handcrafted heuristic** predictions

Metrics: MSE, correlation, and sign agreement (does the heuristic correctly
predict who's winning?).

In [ ]:
# NN predictions on all data
model.eval()
with torch.no_grad():
    nn_preds = model(torch.tensor(X_data)).squeeze().numpy()

# Handcrafted heuristic predictions
hc_preds = []
ai_player, opponent = Board.X, Board.O
for encoded in X_data:
    # Reconstruct board from encoding
    board = Board(3)
    idx = 0
    x_count, o_count = 0, 0
    for r in range(3):
        for c in range(3):
            val = encoded[idx]
            if val > 0.5:
                board.grid[r][c] = ai_player
                x_count += 1
            elif val < -0.5:
                board.grid[r][c] = opponent
                o_count += 1
            idx += 1
    board.move_count = x_count + o_count
    score = evaluate_board(board, ai_player, opponent)
    hc_preds.append(score / 10.0)  # Normalize
hc_preds = np.array(hc_preds)

# Compute metrics
nn_mse = np.mean((nn_preds - y_data) ** 2)
hc_mse = np.mean((hc_preds - y_data) ** 2)
nn_corr = np.corrcoef(nn_preds, y_data)[0, 1]
hc_corr = np.corrcoef(hc_preds, y_data)[0, 1]
nn_sign = np.mean(np.sign(nn_preds) == np.sign(y_data))
hc_sign = np.mean(np.sign(hc_preds) == np.sign(y_data))

print(f"{'Metric':<25} {'Neural Net':<15} {'Handcrafted':<15}")
print("-" * 55)
print(f"{'MSE':<25} {nn_mse:<15.6f} {hc_mse:<15.6f}")
print(f"{'Correlation':<25} {nn_corr:<15.4f} {hc_corr:<15.4f}")
print(f"{'Sign Agreement':<25} {nn_sign:<15.2%} {hc_sign:<15.2%}")

In [ ]:
# Scatter plot: predicted vs true
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(y_data, nn_preds, alpha=0.3, s=10, color='#3498db')
ax1.plot([-1, 1], [-1, 1], 'r--', linewidth=1, label='Perfect prediction')
ax1.set_xlabel('True Minimax Score')
ax1.set_ylabel('NN Predicted Score')
ax1.set_title(f'Neural Network (MSE={nn_mse:.4f}, r={nn_corr:.4f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.scatter(y_data, hc_preds, alpha=0.3, s=10, color='#e67e22')
ax2.plot([-1, 1], [-1, 1], 'r--', linewidth=1, label='Perfect prediction')
ax2.set_xlabel('True Minimax Score')
ax2.set_ylabel('Handcrafted Predicted Score')
ax2.set_title(f'Handcrafted (MSE={hc_mse:.4f}, r={hc_corr:.4f})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Head-to-Head: NN Heuristic vs Handcrafted Heuristic

Pit two `AlphaBetaAgent`s against each other — one using the NN heuristic,
the other using the handcrafted heuristic — on a **4×4 board** with various
depth limits.

In [ ]:
from nn_heuristic import make_nn_heuristic, train_nn

# Train NN for 4x4 board
print("Training NN for 4×4 board (sampled data)...")
np.random.seed(42)
torch.manual_seed(42)
model_4x4, tl_4, vl_4 = train_nn(board_size=4, epochs=300, verbose=True)

nn_heuristic_4x4 = make_nn_heuristic(4)

In [ ]:
# Head-to-head at different depth limits
random.seed(42)
h2h_results = {}

for depth in [2, 4, 6]:
    nn_wins, hc_wins, draws = 0, 0, 0
    num_games = 50

    for game_num in range(num_games):
        board = Board(4)
        # Alternate who goes first
        if game_num % 2 == 0:
            nn_sym, hc_sym = Board.X, Board.O
        else:
            nn_sym, hc_sym = Board.O, Board.X

        nn_agent = AlphaBetaAgent(nn_sym, depth_limit=depth, heuristic=nn_heuristic_4x4)
        hc_agent = AlphaBetaAgent(hc_sym, depth_limit=depth, heuristic=evaluate_board)

        while not board.is_terminal():
            current = board.current_player()
            if current == nn_sym:
                move = nn_agent.get_move(board)
            else:
                move = hc_agent.get_move(board)
            if move:
                board.make_move(move[0], move[1], current)

        winner = board.check_winner()
        if winner == nn_sym:
            nn_wins += 1
        elif winner == hc_sym:
            hc_wins += 1
        else:
            draws += 1

    h2h_results[depth] = {'nn_wins': nn_wins, 'hc_wins': hc_wins, 'draws': draws}
    print(f"Depth {depth}: NN wins={nn_wins}, HC wins={hc_wins}, Draws={draws}")

In [ ]:
# Visualize head-to-head results
fig, ax = plt.subplots(figsize=(8, 5))

depths = sorted(h2h_results.keys())
nn_w = [h2h_results[d]['nn_wins'] for d in depths]
hc_w = [h2h_results[d]['hc_wins'] for d in depths]
dr = [h2h_results[d]['draws'] for d in depths]

x = range(len(depths))
width = 0.25
ax.bar([i - width for i in x], nn_w, width, label='NN Wins', color='#3498db')
ax.bar(x, dr, width, label='Draws', color='#f39c12')
ax.bar([i + width for i in x], hc_w, width, label='HC Wins', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels([f'Depth {d}' for d in depths])
ax.set_ylabel('Games')
ax.set_title('NN Heuristic vs Handcrafted Heuristic (4×4 Board)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 8. Speed Comparison

Compare the average time per move when using the NN heuristic vs the
handcrafted heuristic in Alpha-Beta search.

In [ ]:
random.seed(42)
num_games = 30
depth = 4

speed_results = {}
for label, heuristic_fn in [('Neural Net', nn_heuristic_4x4),
                             ('Handcrafted', evaluate_board)]:
    total_time = 0.0
    total_moves = 0
    wins = 0

    for _ in range(num_games):
        board = Board(4)
        ai = AlphaBetaAgent(Board.X, depth_limit=depth, heuristic=heuristic_fn)
        rand = RandomAgent(Board.O)

        while not board.is_terminal():
            current = board.current_player()
            if current == Board.X:
                move = ai.get_move(board)
                total_time += ai.stats.elapsed_time
                total_moves += 1
            else:
                move = rand.get_move(board)
            if move:
                board.make_move(move[0], move[1], current)
        if board.check_winner() == Board.X:
            wins += 1

    avg_time = total_time / max(total_moves, 1)
    speed_results[label] = {'avg_time': avg_time, 'wins': wins}
    print(f"{label}: avg time/move = {avg_time:.6f}s, wins = {wins}/{num_games}")

nn_t = speed_results['Neural Net']['avg_time']
hc_t = speed_results['Handcrafted']['avg_time']
print(f"\nSpeed ratio (HC/NN): {hc_t / max(nn_t, 1e-9):.2f}x")

In [ ]:
# Speed comparison bar chart
fig, ax = plt.subplots(figsize=(6, 5))

methods = ['Neural Net', 'Handcrafted']
times = [speed_results[m]['avg_time'] * 1000 for m in methods]  # ms
bars = ax.bar(methods, times, color=['#3498db', '#e67e22'])
ax.set_ylabel('Avg Time per Move (ms)')
ax.set_title('Evaluation Speed: NN vs Handcrafted (4×4, depth=4)')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f'{val:.2f}ms', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
## 9. Interactive Play Against NN-Powered AI

Play against the AI using the neural network heuristic.
Change the coordinates in `human_move(row, col)` and re-run the cell for each turn.

In [ ]:
# Initialize a new game with NN-powered AI
nn_heuristic_3x3 = make_nn_heuristic(3)
game_board = Board(3)
nn_ai = AlphaBetaAgent(Board.O, depth_limit=None)  # Full depth for 3x3

def human_move(row, col):
    """Make a human move and let the NN-powered AI respond."""
    if game_board.is_terminal():
        print("Game is already over!")
        return

    if not game_board.make_move(row, col, Board.X):
        print(f"Invalid move ({row}, {col}). Try again.")
        return

    print(f"You played: ({row}, {col})")
    print(game_board)
    print()

    if game_board.is_terminal():
        winner = game_board.check_winner()
        print("You win!" if winner == Board.X else "It's a draw!")
        return

    # AI responds
    move = nn_ai.get_move(game_board)
    game_board.make_move(move[0], move[1], Board.O)
    print(f"AI played: ({move[0]}, {move[1]})")
    print(f"  Nodes explored: {nn_ai.stats.nodes_explored}")
    print(f"  Time: {nn_ai.stats.elapsed_time:.4f}s")
    print()
    print(game_board)

    if game_board.is_terminal():
        winner = game_board.check_winner()
        print()
        if winner == Board.O:
            print("AI wins!")
        elif winner == Board.X:
            print("You win!")
        else:
            print("It's a draw!")

print("Game started! You are X, AI is O (NN-powered).")
print(game_board)
print("\nCall human_move(row, col) to play.")

In [ ]:
# Make your moves here
human_move(1, 1)  # Example: play center

---
## 10. Summary

### What We Built
A **PyTorch feedforward neural network** trained on ground-truth minimax scores
to serve as a drop-in replacement for the handcrafted board evaluation heuristic.

### Key Findings

| Metric | Neural Net | Handcrafted |
|--------|-----------|-------------|
| **Training** | Learns from data (minimax scores) | Manually engineered rules |
| **Accuracy** | Trained to approximate true minimax | Rule-based approximation |
| **Adaptability** | Can retrain for any board size | Rules need manual adjustment |
| **Speed** | Matrix multiply per evaluation | Simple arithmetic |

### Architecture
- **Input**: 10 features (9 board cells + turn indicator)
- **Hidden**: 64 → 32 neurons (ReLU)
- **Output**: 1 scalar (tanh → scaled to [-10, 10])
- **Parameters**: ~2,800 trainable weights
- **Training**: 500 epochs, Adam optimizer, MSE loss

### AI Approaches
- **Search-based**: Minimax and Alpha-Beta pruning (optimal on small boards like 3×3)
- **Heuristic search**: Depth-limited Alpha-Beta with handcrafted evaluation for larger boards
- **Neural network heuristic**: Learned board evaluation trained from minimax-generated data

### Takeaway
The neural network can learn to approximate board position evaluations from data alone,
producing scores that correlate with optimal minimax values. This demonstrates the
potential of **learned evaluation functions** — the same approach (at much larger scale)
is used in AlphaGo and modern game-playing AI systems.